In [1]:
import pandas as pd
import numpy as np
import duckdb
from pathlib import Path

# Project paths
PROJECT_ROOT = Path.cwd().parent

STATIONS_PATH = PROJECT_ROOT / "data" / "raw" / "stations.parquet"
DAILY_PATH = PROJECT_ROOT / "data" / "raw" / "daily.parquet"
HOURLY_PATH = PROJECT_ROOT / "data" / "raw" / "hourly.parquet"

INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

INTERIM_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# DuckDB connection
con = duckdb.connect()

print("Project root:", PROJECT_ROOT)
print("Stations:", STATIONS_PATH)
print("Daily:", DAILY_PATH)
print("Hourly:", HOURLY_PATH)

Project root: D:\01_PROJECTS\DS_AI\flood-prediction-india
Stations: D:\01_PROJECTS\DS_AI\flood-prediction-india\data\raw\stations.parquet
Daily: D:\01_PROJECTS\DS_AI\flood-prediction-india\data\raw\daily.parquet
Hourly: D:\01_PROJECTS\DS_AI\flood-prediction-india\data\raw\hourly.parquet


In [2]:
print(STATIONS_PATH.exists())
print(DAILY_PATH.exists())
print(HOURLY_PATH.exists())

True
True
True


In [3]:
print("STATIONS")
print(
    con.execute(
        f"DESCRIBE SELECT * FROM read_parquet('{STATIONS_PATH}')"
    ).fetchdf()
)

print("\nDAILY")
print(
    con.execute(
        f"DESCRIBE SELECT * FROM read_parquet('{DAILY_PATH}')"
    ).fetchdf()
)

STATIONS
           column_name column_type null   key default extra
0         station_code     VARCHAR  YES  None    None  None
1                 name     VARCHAR  YES  None    None  None
2                  lat      DOUBLE  YES  None    None  None
3                  lon      DOUBLE  YES  None    None  None
4                 type     VARCHAR  YES  None    None  None
5         nearest_town     VARCHAR  YES  None    None  None
6        warning_level      DOUBLE  YES  None    None  None
7         danger_level      DOUBLE  YES  None    None  None
8                  frl      DOUBLE  YES  None    None  None
9                  mwl      DOUBLE  YES  None    None  None
10  highest_flow_level      DOUBLE  YES  None    None  None

DAILY
      column_name column_type null   key default extra
0    station_code     VARCHAR  YES  None    None  None
1   datatype_code     VARCHAR  YES  None    None  None
2        variable     VARCHAR  YES  None    None  None
3            unit     VARCHAR  YES  None    

In [4]:
print("\nHOURLY")
print(
    con.execute(
        f"DESCRIBE SELECT * FROM read_parquet('{HOURLY_PATH}')"
    ).fetchdf()
)


HOURLY
        column_name column_type null   key default extra
0      station_code     VARCHAR  YES  None    None  None
1  station_code_raw     VARCHAR  YES  None    None  None
2          datetime   TIMESTAMP  YES  None    None  None
3     datatype_code     VARCHAR  YES  None    None  None
4          variable     VARCHAR  YES  None    None  None
5              unit     VARCHAR  YES  None    None  None
6             value      DOUBLE  YES  None    None  None
7       source_type     VARCHAR  YES  None    None  None


In [5]:
stations = pd.read_parquet(STATIONS_PATH)

print(stations.shape)
display(stations.head())

(1747, 11)


,station_code,name,lat,lon,type,nearest_town,warning_level,danger_level,frl,mwl,highest_flow_level
0,001-CDBNG,Shimoga,13.926944,75.585000,Base,NaN,566.7,567.25,NaN,NaN,570.18
1,001-CDJPR,Dhareri,23.147500,75.508333,Base,,478.5,479.50,NaN,NaN,482.62
2,001-CHEJMU,Khanabal,33.763889,75.127500,Base,Khanabal,1591.2,1592.50,NaN,NaN,1595.43
3,001-DDASL,Tilaiya Reservoir,24.320000,85.520000,Inflow,,NaN,NaN,372.475,371.465,372.28
4,001-ERDBWN,Balimundali,21.733889,86.633333,Base,NaN,NaN,NaN,NaN,NaN,43.10


In [6]:
stations["danger_level"].notna().sum()

np.int64(1035)

In [7]:
stations["danger_level"].isna().sum()

np.int64(712)

In [8]:
stations[
    ["station_code", "name", "type", "danger_level"]
].head(20)

,station_code,name,type,danger_level
0,001-CDBNG,Shimoga,Base,567.250
1,001-CDJPR,Dhareri,Base,479.500
2,001-CHEJMU,Khanabal,Base,1592.500
3,001-DDASL,Tilaiya Reservoir,Inflow,NaN
4,001-ERDBWN,Balimundali,Base,NaN
5,001-HGDDDN,Badrinath,Base,3113.000
6,001-HYDCHENNAI,Chennur,Base,122.875
7,001-LBDJPG,Mathanguri,Level,99.100
8,001-LGD3BEH,BEDEPARA,Base,NaN
9,001-LGDHYD,Somanpally(Seasonal),Base,NaN


In [9]:
station_parameter_summary = con.execute(f"""
    SELECT
        station_code,
        COUNT(DISTINCT CASE
            WHEN datatype_code = 'HHS'
            THEN date_ist
        END) AS hhs_days,

        COUNT(DISTINCT CASE
            WHEN datatype_code = 'MPS'
            THEN date_ist
        END) AS mps_days

    FROM read_parquet('{DAILY_PATH}')
    WHERE datatype_code IN ('HHS', 'MPS')
    GROUP BY station_code
""").fetchdf()

display(station_parameter_summary.head())

,station_code,hhs_days,mps_days
0,009-LKDHYD,5128,4324
1,009-MGD2LKN,17083,13403
2,009-MGD3VNS,4159,3979
3,018-DDASL,13414,7312
4,006-LYDAGR,11524,15915


In [10]:
station_candidates = stations.merge(
    station_parameter_summary,
    on="station_code",
    how="inner"
)

display(
    station_candidates[
        [
            "station_code",
            "name",
            "type",
            "danger_level",
            "hhs_days",
            "mps_days"
        ]
    ].head(20)
)

,station_code,name,type,danger_level,hhs_days,mps_days
0,001-CDBNG,Shimoga,Base,567.250,12959,12638
1,001-CDJPR,Dhareri,Base,479.500,12483,14234
2,001-CHEJMU,Khanabal,Base,1592.500,8003,6
3,001-DDASL,Tilaiya Reservoir,Inflow,NaN,10683,8319
4,001-ERDBWN,Balimundali,Base,NaN,3956,13781
5,001-HGDDDN,Badrinath,Base,3113.000,7361,7877
6,001-HYDCHENNAI,Chennur,Base,122.875,11447,13106
7,001-LBDJPG,Mathanguri,Level,99.100,5086,4489
8,001-LGD3BEH,BEDEPARA,Base,NaN,12961,1096
9,001-LGDHYD,Somanpally(Seasonal),Base,NaN,3378,1949


In [11]:
candidate_stations = station_candidates[
    station_candidates["danger_level"].notna()
    & (station_candidates["hhs_days"] >= 365 * 3)
    & (station_candidates["mps_days"] >= 365 * 3)
].copy()

print("Initial candidates:", len(candidate_stations))

Initial candidates: 818


In [12]:
overlap = con.execute(f"""
    WITH hhs AS (
        SELECT DISTINCT
            station_code,
            CAST(date_ist AS DATE) AS date
        FROM read_parquet('{DAILY_PATH}')
        WHERE datatype_code = 'HHS'
    ),

    mps AS (
        SELECT DISTINCT
            station_code,
            CAST(date_ist AS DATE) AS date
        FROM read_parquet('{DAILY_PATH}')
        WHERE datatype_code = 'MPS'
    )

    SELECT
        h.station_code,
        COUNT(*) AS overlap_days,
        MIN(h.date) AS overlap_start,
        MAX(h.date) AS overlap_end
    FROM hhs h
    INNER JOIN mps r
        ON h.station_code = r.station_code
       AND h.date = r.date
    GROUP BY h.station_code
""").fetchdf()

display(overlap.head())

,station_code,overlap_days,overlap_start,overlap_end
0,011-MGD1LKN,4750,1979-06-16,2026-09-11
1,011-MGD2LKN,10598,1988-09-21,2026-09-11
2,013-HYDCHENNAI,7289,1989-11-13,2026-09-11
3,024-DDASL,4259,2000-08-30,2026-09-11
4,024-LGDHYD,3710,2014-06-01,2026-09-11


In [13]:
candidate_stations = candidate_stations.merge(
    overlap,
    on="station_code",
    how="inner"
)

In [14]:
candidate_stations = candidate_stations[
    candidate_stations["overlap_days"] >= 365 * 3
].copy()

In [16]:
print("Candidates after overlap:", len(candidate_stations))

display(
    candidate_stations[
        [
            "station_code",
            "name",
            "danger_level",
            "hhs_days",
            "mps_days",
            "overlap_days",
            "overlap_start",
            "overlap_end"
        ]
    ].sort_values("overlap_days", ascending=False).head(30)
)

Candidates after overlap: 813


,station_code,name,danger_level,hhs_days,mps_days,overlap_days,overlap_start,overlap_end
291,014-HGDDDN,Rishikesh,340.50,20318,20034,19987,1971-01-01,2026-09-11
154,008-HGDDDN,Uttarkashi,1123.00,19977,19970,19799,1971-12-27,2026-09-11
23,002-HGDDDN,Joshimath,1383.00,19464,20198,19448,1971-05-01,2026-09-11
114,006-HGDDDN,Rudraprayag (M),626.00,20094,18487,18286,1972-09-01,2026-09-11
270,013-HGDDDN,Marora,556.00,20011,17774,17768,1972-11-01,2026-09-11
236,011-MGD4PTN,Dumariaghat,62.22,18162,17690,17689,1978-01-02,2026-09-11
20,0010-LYDAGRA,SAHIJINA,104.54,20738,19218,17661,1964-07-10,2026-09-11
214,010-UYDDEL,Kalanaur,267.20,17749,18711,17636,1975-11-01,2026-09-11
539,028-MGD1LKN,Jauljibi,607.30,17650,18003,17541,1977-01-01,2026-09-11
353,017-HGDDDN,Karnaprayag (P),773.00,18115,17540,17518,1978-07-01,2026-09-11


In [17]:
con.register("candidate_stations_df", candidate_stations[["station_code"]])

In [18]:
hhs_quality = con.execute(f"""
    SELECT
        d.station_code,

        COUNT(*) AS total_rows,

        COUNT(*) FILTER (
            WHERE d.mean IS NULL
        ) AS null_mean,

        COUNT(*) FILTER (
            WHERE d.mean IN (-999, -9990)
        ) AS sentinel_mean,

        COUNT(*) FILTER (
            WHERE d.mean < -1000
        ) AS extreme_negative,

        COUNT(*) FILTER (
            WHERE d.mean > 10000
        ) AS extreme_high,

        quantile_cont(d.mean, 0.01) AS p01,
        quantile_cont(d.mean, 0.50) AS median,
        quantile_cont(d.mean, 0.99) AS p99,
        MAX(d.mean) AS max_value

    FROM read_parquet('{DAILY_PATH}') d

    INNER JOIN candidate_stations_df c
        ON d.station_code = c.station_code

    WHERE d.datatype_code = 'HHS'

    GROUP BY d.station_code
""").fetchdf()

display(hhs_quality.head())

,station_code,total_rows,null_mean,sentinel_mean,extreme_negative,extreme_high,p01,median,p99,max_value
0,006-UBDDIB,12201,0,0,0,0,185.210000,191.050000,639.433333,642.906667
1,025-MGD5PTN,17193,0,0,0,0,139.100000,140.954167,143.328267,183.617375
2,007-TDSURAT,4260,0,0,0,0,379.461737,391.299972,398.070000,1941.738000
3,008-CDJAPR,4082,0,0,0,0,52.500000,420.051875,422.668887,428.315000
4,008-ERDBWN,4069,0,0,0,0,115.000000,115.320000,120.581767,123.745833


In [19]:
display(
    hhs_quality.describe()
)

,total_rows,null_mean,sentinel_mean,extreme_negative,extreme_high,p01,median,p99,max_value
count,813.000000,813.0,813.000000,813.000000,813.000000,813.000000,813.000000,813.000000,813.000000
mean,7309.146371,0.0,10.241082,0.001230,0.163592,173.375294,230.487660,250.839139,5796.950463
std,5729.784777,0.0,281.843738,0.035072,1.054688,315.894314,327.832455,347.648505,39318.993417
min,1173.000000,0.0,0.000000,0.000000,0.000000,-999.000000,-999.000000,0.800000,2.000000
25%,2899.000000,0.0,0.000000,0.000000,0.000000,15.500000,49.600000,56.714750,86.315833
50%,4251.000000,0.0,0.000000,0.000000,0.000000,80.651300,123.583333,134.875433,247.708889
75%,10874.000000,0.0,0.000000,0.000000,0.000000,202.190000,287.560000,308.750658,813.836667
max,25787.000000,0.0,8034.000000,1.000000,20.000000,3110.428667,3111.213333,3111.747333,663559.513333


In [20]:
display(
    hhs_quality.sort_values(
        "extreme_high",
        ascending=False
    ).head(30)
)

,station_code,total_rows,null_mean,sentinel_mean,extreme_negative,extreme_high,p01,median,p99,max_value
452,017-MGD2LKN,20463,0,0,0,20,195.028747,196.270000,198.738530,663559.513333
437,007-DDASL,12887,0,0,0,12,302.340000,303.330000,306.797492,304240.000000
74,019-CDJAPR,11675,0,0,0,8,0.000000,386.930000,391.278483,129183.853333
328,003-HYDCHENNAI,4367,0,0,0,8,0.000000,96.328148,191.222149,32197.600000
537,016-MGD2LKN,19006,0,0,0,7,94.940000,95.914000,101.797850,33705.266000
179,011-MGD2LKN,19799,0,0,0,6,118.350000,119.800000,123.681485,40909.921667
105,001-HYDCHENNAI,11447,0,0,0,5,112.570767,117.285000,218.742778,19514.595833
303,011-DDASL,12248,0,0,0,5,409.149508,422.209000,426.550031,423687.000000
429,020-CDJAPR,7271,0,0,0,5,0.000000,412.439583,414.520125,14058.660000
248,039-MGD3VNS,2570,0,0,0,4,58.281050,61.720000,92.926867,30644.450000


In [21]:
display(
    hhs_quality.sort_values(
        "sentinel_mean",
        ascending=False
    ).head(30)
)

,station_code,total_rows,null_mean,sentinel_mean,extreme_negative,extreme_high,p01,median,p99,max_value
38,017-CDJAPR,15588,0,8034,0,0,-999.000000,-999.000000,302.977617,312.649583
503,024-MGD5PAT,14275,0,200,0,0,-999.000000,207.080000,208.554149,1065.393333
526,047-MGD1LKN,2883,0,28,0,0,-65.797607,89.580000,95.749767,156.136667
718,001-MGD3VNS,4003,0,28,0,0,65.542242,66.250000,72.545133,73.714167
99,009-NDBHP,3996,0,21,0,0,442.570396,443.440000,447.911063,453.880417
728,010MGD5PTN,8242,0,5,0,0,40.016947,48.720979,51.176415,52.025833
211,013-TDSURAT,4182,0,3,0,0,462.000000,462.000000,463.748239,1828.666667
65,022-MGD5PTN,16707,0,3,0,0,122.750000,123.583333,126.109017,639.936250
401,005-SRDCBE,16076,0,1,0,0,338.335000,338.734375,339.556018,483.012857
507,039-MGD4PTN,2541,0,1,1,0,64.000000,65.222083,68.023833,87.890000


In [22]:
from pathlib import Path
import pandas as pd
import numpy as np
import duckdb

PROJECT_ROOT = Path.cwd().parent

STATIONS_PATH = PROJECT_ROOT / "data" / "raw" / "stations.parquet"
DAILY_PATH = PROJECT_ROOT / "data" / "raw" / "daily.parquet"
HOURLY_PATH = PROJECT_ROOT / "data" / "raw" / "hourly.parquet"

con = duckdb.connect()

print("Project root:", PROJECT_ROOT)
print("Stations:", STATIONS_PATH.exists(), STATIONS_PATH)
print("Daily:", DAILY_PATH.exists(), DAILY_PATH)
print("Hourly:", HOURLY_PATH.exists(), HOURLY_PATH)

Project root: D:\01_PROJECTS\DS_AI\flood-prediction-india
Stations: True D:\01_PROJECTS\DS_AI\flood-prediction-india\data\raw\stations.parquet
Daily: True D:\01_PROJECTS\DS_AI\flood-prediction-india\data\raw\daily.parquet
Hourly: True D:\01_PROJECTS\DS_AI\flood-prediction-india\data\raw\hourly.parquet


In [23]:
stations = pd.read_parquet(STATIONS_PATH)

print("Station shape:", stations.shape)
display(stations.head())

Station shape: (1747, 11)


,station_code,name,lat,lon,type,nearest_town,warning_level,danger_level,frl,mwl,highest_flow_level
0,001-CDBNG,Shimoga,13.926944,75.585000,Base,NaN,566.7,567.25,NaN,NaN,570.18
1,001-CDJPR,Dhareri,23.147500,75.508333,Base,,478.5,479.50,NaN,NaN,482.62
2,001-CHEJMU,Khanabal,33.763889,75.127500,Base,Khanabal,1591.2,1592.50,NaN,NaN,1595.43
3,001-DDASL,Tilaiya Reservoir,24.320000,85.520000,Inflow,,NaN,NaN,372.475,371.465,372.28
4,001-ERDBWN,Balimundali,21.733889,86.633333,Base,NaN,NaN,NaN,NaN,NaN,43.10


In [24]:
print(
    con.execute(
        f"DESCRIBE SELECT * FROM read_parquet('{DAILY_PATH}')"
    ).fetchdf()
)

      column_name column_type null   key default extra
0    station_code     VARCHAR  YES  None    None  None
1   datatype_code     VARCHAR  YES  None    None  None
2        variable     VARCHAR  YES  None    None  None
3            unit     VARCHAR  YES  None    None  None
4     source_type     VARCHAR  YES  None    None  None
5        date_ist        DATE  YES  None    None  None
6             min      DOUBLE  YES  None    None  None
7            mean      DOUBLE  YES  None    None  None
8             max      DOUBLE  YES  None    None  None
9             sum      DOUBLE  YES  None    None  None
10          n_obs    UINTEGER  YES  None    None  None


In [25]:
station_parameter_summary = con.execute(f"""
    SELECT
        station_code,

        COUNT(DISTINCT CASE
            WHEN datatype_code = 'HHS'
            THEN date_ist
        END) AS hhs_days,

        COUNT(DISTINCT CASE
            WHEN datatype_code = 'MPS'
            THEN date_ist
        END) AS mps_days

    FROM read_parquet('{DAILY_PATH}')

    WHERE datatype_code IN ('HHS', 'MPS')

    GROUP BY station_code
""").fetchdf()

print("Stations with HHS or MPS:", len(station_parameter_summary))

display(station_parameter_summary.head(20))

Stations with HHS or MPS: 1625


,station_code,hhs_days,mps_days
0,009-HYDCHENNAI,8808,14189
1,009-LBDJPG,7909,1264
2,009-SWRDKOCHI,5084,944
3,017-LKDHYD,4330,4471
4,017-MGD2LKN,20463,10991
5,017-UYDDEL,11831,14669
6,01-10-21-001,2497,29
7,018-LYDAGRA,11051,14952
8,018-MGD3VNS,8050,3800
9,018-WGDNGP,3529,3449


In [26]:
station_candidates = stations.merge(
    station_parameter_summary,
    on="station_code",
    how="inner"
)

print("Stations found in daily HHS/MPS data:", len(station_candidates))

display(
    station_candidates[
        [
            "station_code",
            "name",
            "type",
            "danger_level",
            "hhs_days",
            "mps_days"
        ]
    ].head(30)
)

Stations found in daily HHS/MPS data: 1623


,station_code,name,type,danger_level,hhs_days,mps_days
0,001-CDBNG,Shimoga,Base,567.250,12959,12638
1,001-CDJPR,Dhareri,Base,479.500,12483,14234
2,001-CHEJMU,Khanabal,Base,1592.500,8003,6
3,001-DDASL,Tilaiya Reservoir,Inflow,NaN,10683,8319
4,001-ERDBWN,Balimundali,Base,NaN,3956,13781
5,001-HGDDDN,Badrinath,Base,3113.000,7361,7877
6,001-HYDCHENNAI,Chennur,Base,122.875,11447,13106
7,001-LBDJPG,Mathanguri,Level,99.100,5086,4489
8,001-LGD3BEH,BEDEPARA,Base,NaN,12961,1096
9,001-LGDHYD,Somanpally(Seasonal),Base,NaN,3378,1949


In [27]:
candidate_stations = station_candidates[
    station_candidates["danger_level"].notna()
    & (station_candidates["hhs_days"] >= 365 * 3)
    & (station_candidates["mps_days"] >= 365 * 3)
].copy()

print("Initial candidate stations:", len(candidate_stations))

display(
    candidate_stations[
        [
            "station_code",
            "name",
            "type",
            "danger_level",
            "hhs_days",
            "mps_days"
        ]
    ].head(30)
)

Initial candidate stations: 818


,station_code,name,type,danger_level,hhs_days,mps_days
0,001-CDBNG,Shimoga,Base,567.250,12959,12638
1,001-CDJPR,Dhareri,Base,479.500,12483,14234
5,001-HGDDDN,Badrinath,Base,3113.000,7361,7877
6,001-HYDCHENNAI,Chennur,Base,122.875,11447,13106
7,001-LBDJPG,Mathanguri,Level,99.100,5086,4489
11,001-LYDAGRA,AGRA (J.B.),Level,152.400,12787,2746
12,001-MAHGAND,Chakaliya,Base,225.000,11148,4267
13,001-MBDGHY,Guwahati(D.C.Court),Level,49.680,6395,3747
15,001-MDSIL,KUMARGHAT GD SITE,Base,30.000,8091,12008
16,001-MGD1LKN,Colonelganj,Base,105.000,20180,4533


In [28]:
overlap = con.execute(f"""
    WITH hhs AS (
        SELECT DISTINCT
            station_code,
            date_ist AS date
        FROM read_parquet('{DAILY_PATH}')
        WHERE datatype_code = 'HHS'
    ),

    mps AS (
        SELECT DISTINCT
            station_code,
            date_ist AS date
        FROM read_parquet('{DAILY_PATH}')
        WHERE datatype_code = 'MPS'
    )

    SELECT
        h.station_code,

        COUNT(*) AS overlap_days,

        MIN(h.date) AS overlap_start,

        MAX(h.date) AS overlap_end

    FROM hhs h

    INNER JOIN mps r
        ON h.station_code = r.station_code
       AND h.date = r.date

    GROUP BY h.station_code
""").fetchdf()

print("Stations with HHS-MPS overlap:", len(overlap))

display(
    overlap.sort_values(
        "overlap_days",
        ascending=False
    ).head(30)
)

Stations with HHS-MPS overlap: 1507


,station_code,overlap_days,overlap_start,overlap_end
299,014-HGDDDN,19987,1971-01-01,2026-09-11
631,008-HGDDDN,19799,1971-12-27,2026-09-11
1364,002-HGDDDN,19448,1971-05-01,2026-09-11
708,006-HGDDDN,18286,1972-09-01,2026-09-11
11,013-HGDDDN,17768,1972-11-01,2026-09-11
1113,011-MGD4PTN,17689,1978-01-02,2026-09-11
495,0010-LYDAGRA,17661,1964-07-10,2026-09-11
1333,010-UYDDEL,17636,1975-11-01,2026-09-11
1225,028-MGD1LKN,17541,1977-01-01,2026-09-11
1381,017-HGDDDN,17518,1978-07-01,2026-09-11


In [29]:
candidate_stations = candidate_stations.merge(
    overlap,
    on="station_code",
    how="inner"
)

candidate_stations = candidate_stations[
    candidate_stations["overlap_days"] >= 365 * 3
].copy()

print("Final candidates after 3-year overlap:", len(candidate_stations))

Final candidates after 3-year overlap: 813


In [30]:
display(
    candidate_stations[
        [
            "station_code",
            "name",
            "type",
            "danger_level",
            "hhs_days",
            "mps_days",
            "overlap_days",
            "overlap_start",
            "overlap_end"
        ]
    ]
    .sort_values("overlap_days", ascending=False)
    .head(50)
)

,station_code,name,type,danger_level,hhs_days,mps_days,overlap_days,overlap_start,overlap_end
291,014-HGDDDN,Rishikesh,Level,340.500,20318,20034,19987,1971-01-01,2026-09-11
154,008-HGDDDN,Uttarkashi,Base,1123.000,19977,19970,19799,1971-12-27,2026-09-11
23,002-HGDDDN,Joshimath,Base,1383.000,19464,20198,19448,1971-05-01,2026-09-11
114,006-HGDDDN,Rudraprayag (M),Base,626.000,20094,18487,18286,1972-09-01,2026-09-11
270,013-HGDDDN,Marora,Base,556.000,20011,17774,17768,1972-11-01,2026-09-11
236,011-MGD4PTN,Dumariaghat,Level,62.220,18162,17690,17689,1978-01-02,2026-09-11
20,0010-LYDAGRA,SAHIJINA,Level,104.540,20738,19218,17661,1964-07-10,2026-09-11
214,010-UYDDEL,Kalanaur,Base,267.200,17749,18711,17636,1975-11-01,2026-09-11
539,028-MGD1LKN,Jauljibi,Base,607.300,17650,18003,17541,1977-01-01,2026-09-11
353,017-HGDDDN,Karnaprayag (P),Base,773.000,18115,17540,17518,1978-07-01,2026-09-11


In [31]:
con.register(
    "candidate_stations_df",
    station_candidates[["station_code", "danger_level"]]
)

In [32]:
hhs_quality = con.execute(f"""
    SELECT
        d.station_code,

        c.danger_level,

        COUNT(*) AS total_rows,

        COUNT(*) FILTER (
            WHERE d.mean IS NULL
        ) AS null_mean,

        COUNT(*) FILTER (
            WHERE d.mean IN (-999, -9990)
        ) AS sentinel_count,

        COUNT(*) FILTER (
            WHERE d.mean < -1000
        ) AS extreme_negative_count,

        COUNT(*) FILTER (
            WHERE d.mean > 10000
        ) AS extreme_high_count,

        quantile_cont(d.mean, 0.01) AS p01,
        quantile_cont(d.mean, 0.50) AS median,
        quantile_cont(d.mean, 0.90) AS p90,
        quantile_cont(d.mean, 0.99) AS p99,

        MAX(d.mean) AS max_value

    FROM read_parquet('{DAILY_PATH}') d

    INNER JOIN candidate_stations_df c
        ON d.station_code = c.station_code

    WHERE d.datatype_code = 'HHS'

    GROUP BY
        d.station_code,
        c.danger_level
""").fetchdf()

print("Stations evaluated:", len(hhs_quality))

display(hhs_quality.head(30))

Stations evaluated: 1607


,station_code,danger_level,total_rows,null_mean,sentinel_count,extreme_negative_count,extreme_high_count,p01,median,p90,p99,max_value
0,002-UBDDIB,441.00,7193,0,0,0,0,411.630000,420.827619,427.288250,430.803783,435.662917
1,003-HGDDDN,1268.00,9098,0,0,0,0,1081.480000,1262.670000,1264.053444,1266.068617,1613.556667
2,019MAHGAND,92.00,4206,0,0,0,0,84.786833,87.000000,87.349444,89.000000,92.066667
3,020-CDJAPR,421.00,7271,0,0,0,5,0.000000,412.439583,413.530000,414.520125,14058.660000
4,020-LBDJPG,NaN,6811,0,0,0,0,157.120000,159.536957,160.333333,160.912250,161.566667
5,013-DDASL,NaN,10305,0,0,0,0,57.895500,64.465000,64.465000,211.436500,449.437500
6,013-LBDJPG,NaN,8469,0,0,0,0,288.133333,292.313750,294.051500,294.751917,295.921250
7,013-MBDGHY,26.20,3863,0,0,0,0,15.148733,17.896667,23.012167,26.304150,30.662500
8,013-UYDDEL,216.50,17196,0,0,0,0,210.000000,212.467500,213.838750,215.242104,216.970000
9,020-MGD1LKN,78.16,20168,0,0,0,0,69.143333,70.562542,73.953583,75.928891,78.113750


In [33]:
hhs_quality["median_danger_ratio"] = (
    hhs_quality["median"] /
    hhs_quality["danger_level"]
)

hhs_quality["p99_danger_ratio"] = (
    hhs_quality["p99"] /
    hhs_quality["danger_level"]
)

hhs_quality["max_danger_ratio"] = (
    hhs_quality["max_value"] /
    hhs_quality["danger_level"]
)

hhs_quality["p99_median_ratio"] = (
    hhs_quality["p99"] /
    hhs_quality["median"].replace(0, np.nan)
)

In [34]:
display(
    hhs_quality[
        [
            "median_danger_ratio",
            "p99_danger_ratio",
            "max_danger_ratio",
            "p99_median_ratio"
        ]
    ].describe()
)

,median_danger_ratio,p99_danger_ratio,max_danger_ratio,p99_median_ratio
count,1026.000000,1026.000000,1026.000000,1602.000000
mean,0.908453,1.094582,25.453098,1.395198
std,0.297033,0.806623,184.016869,2.898810
min,-3.233010,0.014251,0.023116,-1.032258
25%,0.901786,0.982511,1.003618,1.008253
50%,0.966026,0.995417,1.081198,1.023192
75%,0.985692,1.002436,3.817780,1.095869
max,6.404861,11.694419,3328.949548,73.337209


In [35]:
display(
    hhs_quality
    .sort_values("max_danger_ratio", ascending=False)
    [
        [
            "station_code",
            "danger_level",
            "total_rows",
            "median",
            "p99",
            "max_value",
            "median_danger_ratio",
            "p99_danger_ratio",
            "max_danger_ratio"
        ]
    ]
    .head(30)
)

,station_code,danger_level,total_rows,median,p99,max_value,median_danger_ratio,p99_danger_ratio,max_danger_ratio
143,017-MGD2LKN,199.33,20463,196.270000,198.738530,663559.513333,0.984649,0.997033,3328.949548
617,002-MGD2LKN,162.40,18620,159.573333,162.448333,534673.124667,0.982594,1.000298,3292.322196
38,008-LGD3BEH,17.22,20136,14.800000,16.755850,50297.236667,0.859466,0.973046,2920.861595
1489,011-DDASL,427.94,12248,422.209000,426.550031,423687.000000,0.986608,0.996752,990.061691
1395,007-DDASL,307.51,12887,303.330000,306.797492,304240.000000,0.986407,0.997683,989.366199
1329,039-MGD3VNS,65.50,2570,61.720000,92.926867,30644.450000,0.942290,1.418731,467.854198
1059,GGQ00H8,46.40,5751,44.590000,48.741167,19214.935042,0.960991,1.050456,414.114979
1098,008-MGD4PTN,109.67,25766,105.160000,109.225333,36901.011333,0.958877,0.995945,336.473159
1529,027-UYDDEL,1200.00,2633,1196.733333,1202.450317,401478.206667,0.997278,1.002042,334.565172
1477,016-MGD2LKN,101.00,19006,95.914000,101.797850,33705.266000,0.949644,1.007900,333.715505


In [36]:
display(
    hhs_quality
    .sort_values("p99_danger_ratio", ascending=False)
    [
        [
            "station_code",
            "danger_level",
            "total_rows",
            "median",
            "p99",
            "max_value",
            "median_danger_ratio",
            "p99_danger_ratio",
            "max_danger_ratio"
        ]
    ]
    .head(30)
)

,station_code,danger_level,total_rows,median,p99,max_value,median_danger_ratio,p99_danger_ratio,max_danger_ratio
698,037-SWRDKOCHI,8.000,2877,5.286667,93.555350,479.778696,0.660833,11.694419,59.972337
943,010214003,9.000,2599,4.075417,99.647025,131.000000,0.452824,11.071892,14.555556
1169,010214001,8.500,2571,3.686667,91.224625,185.660000,0.433725,10.732309,21.842353
1020,041-SWRDKOCHI,9.600,2879,7.037083,98.091825,199.270000,0.733030,10.217898,20.757292
707,042-SWRDKOCHI,10.000,2499,8.293750,92.167683,96.000000,0.829375,9.216768,9.600000
1332,040-SWRDKOCHI,12.000,2884,10.206667,98.467246,844.206923,0.850556,8.205604,70.350577
1344,049 -HD CHENNAI,14.400,2584,92.230000,95.852938,105.440000,6.404861,6.656454,7.322222
1505,013-HGDDDN,556.000,20011,549.664000,3111.586333,184170.333000,0.988604,5.596378,331.241606
243,010214002,22.000,2596,14.516667,93.367250,128.756667,0.659848,4.243966,5.852576
471,025-MGD3VNS,208.000,3972,202.750000,802.742900,802.850000,0.974760,3.859341,3.859856


In [37]:
display(
    hhs_quality
    .sort_values("sentinel_count", ascending=False)
    [
        [
            "station_code",
            "danger_level",
            "total_rows",
            "sentinel_count",
            "extreme_negative_count",
            "median",
            "p99",
            "max_value"
        ]
    ]
    .head(30)
)

,station_code,danger_level,total_rows,sentinel_count,extreme_negative_count,median,p99,max_value
932,017-CDJAPR,309.00,15588,8034,0,-999.000000,302.977617,312.649583
93,024-MGD5PAT,209.00,14275,200,0,207.080000,208.554149,1065.393333
409,041-CDJAPR,NaN,1214,103,0,255.250000,258.450000,258.620000
297,047-MGD1LKN,95.55,2883,28,0,89.580000,95.749767,156.136667
891,001-MGD3VNS,74.88,4003,28,0,66.250000,72.545133,73.714167
524,049-MGD1LKN,NaN,2824,25,0,96.610208,100.634687,101.405833
1269,NBO-1-044-1245,303.00,2764,24,0,298.900000,301.950000,302.600000
1474,009-NDBHP,462.15,3996,21,0,443.440000,447.911063,453.880417
1025,0010NEID3,NaN,2467,19,0,179.490000,180.664058,924.016667
1175,NBO-2-051-1252,201.00,2741,16,0,198.050000,199.618333,203.190833


In [38]:
quality = hhs_quality.copy()

quality["sentinel_rate"] = (
    quality["sentinel_count"] / quality["total_rows"]
)

quality["eligible_quality"] = (
    quality["danger_level"].notna()
    & quality["median_danger_ratio"].between(0.70, 1.30)
    & quality["p99_danger_ratio"].between(0.75, 1.25)
    & (quality["sentinel_rate"] <= 0.01)
)

print("Total stations:", len(quality))
print(
    "Quality-eligible stations:",
    quality["eligible_quality"].sum()
)

print(
    "Excluded by median ratio:",
    (~quality["median_danger_ratio"].between(0.70, 1.30)).sum()
)

print(
    "Excluded by p99 ratio:",
    (~quality["p99_danger_ratio"].between(0.75, 1.25)).sum()
)

print(
    "Excluded by sentinel rate:",
    (quality["sentinel_rate"] > 0.01).sum()
)

Total stations: 1607
Quality-eligible stations: 894
Excluded by median ratio: 677
Excluded by p99 ratio: 658
Excluded by sentinel rate: 3


In [39]:
quality.loc[
    quality["eligible_quality"],
    [
        "station_code",
        "danger_level",
        "total_rows",
        "sentinel_count",
        "sentinel_rate",
        "median_danger_ratio",
        "p99_danger_ratio",
        "max_danger_ratio"
    ]
].head(20)

,station_code,danger_level,total_rows,sentinel_count,sentinel_rate,median_danger_ratio,p99_danger_ratio,max_danger_ratio
0,002-UBDDIB,441.000,7193,0,0.0,0.954258,0.976879,0.987898
1,003-HGDDDN,1268.000,9098,0,0.0,0.995797,0.998477,1.272521
2,019MAHGAND,92.000,4206,0,0.0,0.945652,0.967391,1.000725
3,020-CDJAPR,421.000,7271,0,0.0,0.979666,0.984608,33.393492
8,013-UYDDEL,216.500,17196,0,0.0,0.981374,0.994190,1.002171
9,020-MGD1LKN,78.160,20168,0,0.0,0.902796,0.971455,0.999408
11,006-UYDDEL,901.000,16930,0,0.0,0.993962,0.995468,1.031091
13,018-UKDPUNE,457.600,3269,0,0.0,0.987762,0.992281,0.993475
14,019-CDBNG,608.000,7203,0,0.0,0.994458,0.996592,1.000744
16,024-UGDHYD,316.630,3687,0,0.0,0.982731,0.993389,1.152699


In [40]:
quality["sentinel_rate"].describe()

count    1607.000000
mean        0.000423
std         0.013042
min         0.000000
25%         0.000000
50%         0.000000
75%         0.000000
max         0.515396
Name: sentinel_rate, dtype: float64

In [41]:
# Add sentinel rate
hhs_quality["sentinel_rate"] = (
    hhs_quality["sentinel_count"] /
    hhs_quality["total_rows"]
)

# Lightweight quality filter
usable_quality = hhs_quality[
    hhs_quality["danger_level"].notna()
    & hhs_quality["median_danger_ratio"].between(0.70, 1.30)
    & hhs_quality["p99_danger_ratio"].between(0.75, 1.25)
    & (hhs_quality["sentinel_rate"] <= 0.01)
].copy()

print("Quality-eligible:", len(usable_quality))

Quality-eligible: 894


In [42]:
final_stations = station_candidates.merge(
    usable_quality[
        [
            "station_code",
            "sentinel_rate",
            "median_danger_ratio",
            "p99_danger_ratio"
        ]
    ],
    on="station_code",
    how="inner"
)

print("Final usable stations:", len(final_stations))

Final usable stations: 894


In [43]:
final_stations.head()

,station_code,name,lat,lon,type,nearest_town,warning_level,danger_level,frl,mwl,highest_flow_level,hhs_days,mps_days,sentinel_rate,median_danger_ratio,p99_danger_ratio
0,001-CDBNG,Shimoga,13.926944,75.585000,Base,NaN,566.7,567.25,NaN,NaN,570.18,12959,12638,0.0,0.985203,0.997113
1,001-CDJPR,Dhareri,23.147500,75.508333,Base,,478.5,479.50,NaN,NaN,482.62,12483,14234,0.0,0.972614,0.983106
2,001-CHEJMU,Khanabal,33.763889,75.127500,Base,Khanabal,1591.2,1592.50,NaN,NaN,1595.43,8003,6,0.0,0.997262,0.998926
3,001-HGDDDN,Badrinath,30.748333,79.495833,Base,NaN,3112.0,3113.00,NaN,NaN,3112.70,7361,7877,0.0,0.999426,0.999598
4,001-LBDJPG,Mathanguri,26.782778,90.956667,Level,NaN,98.1,99.10,NaN,NaN,100.28,5086,4489,0.0,0.936245,0.964103


In [44]:
final_stations.shape

(894, 16)

In [45]:
FINAL_STATIONS_PATH = (
    INTERIM_DIR / "final_station_list.parquet"
)

final_stations.to_parquet(
    FINAL_STATIONS_PATH,
    index=False
)

print(FINAL_STATIONS_PATH)

D:\01_PROJECTS\DS_AI\flood-prediction-india\data\interim\final_station_list.parquet


In [48]:
print(station_candidates.columns.tolist())

['station_code', 'name', 'lat', 'lon', 'type', 'nearest_town', 'warning_level', 'danger_level', 'frl', 'mwl', 'highest_flow_level', 'hhs_days', 'mps_days']


In [47]:
print(final_stations.columns.tolist())

['station_code', 'name', 'lat', 'lon', 'type', 'nearest_town', 'warning_level', 'danger_level', 'frl', 'mwl', 'highest_flow_level', 'hhs_days', 'mps_days', 'sentinel_rate', 'median_danger_ratio', 'p99_danger_ratio']


In [49]:
overlap_813 = con.execute(f"""
    SELECT
        d.station_code,
        COUNT(DISTINCT d.date_ist) AS overlap_days
    FROM read_parquet('{DAILY_PATH}') d
    INNER JOIN candidate_stations_df c
        ON d.station_code = c.station_code
    WHERE d.datatype_code IN ('HHS', 'MPS')
    GROUP BY d.station_code
    HAVING
        COUNT(DISTINCT CASE
            WHEN d.datatype_code = 'HHS' THEN d.date_ist
        END) > 0
        AND
        COUNT(DISTINCT CASE
            WHEN d.datatype_code = 'MPS' THEN d.date_ist
        END) > 0
""").fetchdf()

print(overlap_813.shape)
overlap_813.head()

(1532, 2)


,station_code,overlap_days
0,018-UBDDIB,12428
1,018-UYDDEL,7595
2,018MAHGAND,4564
3,019-HGDDDN,2912
4,019-LGDHYD,4534


In [50]:
overlap_813 = con.execute(f"""
    SELECT
        h.station_code,
        COUNT(*) AS overlap_days
    FROM (
        SELECT DISTINCT
            station_code,
            date_ist
        FROM read_parquet('{DAILY_PATH}')
        WHERE datatype_code = 'HHS'
    ) h
    INNER JOIN (
        SELECT DISTINCT
            station_code,
            date_ist
        FROM read_parquet('{DAILY_PATH}')
        WHERE datatype_code = 'MPS'
    ) r
        ON h.station_code = r.station_code
        AND h.date_ist = r.date_ist
    INNER JOIN candidate_stations_df c
        ON h.station_code = c.station_code
    GROUP BY h.station_code
""").fetchdf()

print("Stations:", len(overlap_813))
print(overlap_813["overlap_days"].describe())

Stations: 1506
count     1506.000000
mean      3597.321381
std       3348.664195
min          1.000000
25%       1759.250000
50%       2628.000000
75%       4078.750000
max      19987.000000
Name: overlap_days, dtype: float64


In [51]:
final_stations = final_stations.drop(
    columns=["overlap_days"],
    errors="ignore"
).merge(
    overlap_813,
    on="station_code",
    how="inner"
)

In [52]:
final_stations = final_stations[
    (final_stations["hhs_days"] >= 1095) &
    (final_stations["mps_days"] >= 1095) &
    (final_stations["overlap_days"] >= 1095)
].copy()

In [53]:
print("FINAL usable stations:", len(final_stations))
print("Minimum HHS days:", final_stations["hhs_days"].min())
print("Minimum MPS days:", final_stations["mps_days"].min())
print("Minimum overlap days:", final_stations["overlap_days"].min())

FINAL usable stations: 703
Minimum HHS days: 1173
Minimum MPS days: 1142
Minimum overlap days: 1134


In [54]:
final_stations[
    final_stations["station_code"] == "001-CHEJMU"
]

,station_code,name,lat,lon,type,nearest_town,warning_level,danger_level,frl,mwl,highest_flow_level,hhs_days,mps_days,sentinel_rate,median_danger_ratio,p99_danger_ratio,overlap_days


In [55]:
FINAL_STATIONS_PATH = (
    INTERIM_DIR / "final_station_list.parquet"
)

final_stations.to_parquet(
    FINAL_STATIONS_PATH,
    index=False
)

print(f"Saved {len(final_stations)} stations")
print(FINAL_STATIONS_PATH)

Saved 703 stations
D:\01_PROJECTS\DS_AI\flood-prediction-india\data\interim\final_station_list.parquet


In [56]:
final_stations[
    [
        "station_code",
        "name",
        "danger_level",
        "warning_level",
        "hhs_days",
        "mps_days",
        "overlap_days",
        "sentinel_rate",
        "median_danger_ratio",
        "p99_danger_ratio"
    ]
].head(10)

,station_code,name,danger_level,warning_level,hhs_days,mps_days,overlap_days,sentinel_rate,median_danger_ratio,p99_danger_ratio
0,001-CDBNG,Shimoga,567.25,566.70,12959,12638,12399,0.000000,0.985203,0.997113
1,001-CDJPR,Dhareri,479.50,478.50,12483,14234,12032,0.000000,0.972614,0.983106
3,001-HGDDDN,Badrinath,3113.00,3112.00,7361,7877,7071,0.000000,0.999426,0.999598
4,001-LBDJPG,Mathanguri,99.10,98.10,5086,4489,4487,0.000000,0.936245,0.964103
5,001-LYDAGRA,AGRA (J.B.),152.40,151.40,12787,2746,2741,0.000000,0.962861,0.988439
6,001-MAHGAND,Chakaliya,225.00,223.50,11148,4267,4091,0.000000,0.969467,0.977408
7,001-MBDGHY,Guwahati(D.C.Court),49.68,48.68,6395,3747,3689,0.000000,0.893552,1.003529
8,001-MDSIL,KUMARGHAT GD SITE,30.00,29.00,8091,12008,7471,0.000000,0.879396,0.982956
9,001-MGD1LKN,Colonelganj,105.00,104.00,20180,4533,4452,0.000000,0.960486,0.991252
10,001-MGD3VNS,Azamgarh,74.88,73.88,4003,3956,3946,0.006995,0.884749,0.968819
